# 🚀 Pelatihan (Fine-Tuning) IndoBERT Multi-Task untuk Emotext-CRM
Notebook ini dirancang khusus untuk dijalankan di **Google Colab** agar memanfaatkan GPU Gratis (T4 GPU).
**Langkah 1:** Pastikan Anda menyalakan GPU (Klik menu `Runtime` > `Change runtime type` > Pilih `T4 GPU`).


In [ ]:
# 1. Instalasi Library yang Dibutuhkan
!pip install -q transformers torch pandas scikit-learn onnx


In [ ]:
# 2. Upload Dataset
# Silakan upload file `master_dataset.csv` yang berisi 2.155 baris data (text, intent, sentiment).
from google.colab import files
print("Silakan Upload file master_dataset.csv:")
uploaded = files.upload()


In [ ]:
# 3. Import & Persiapan Data
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
import numpy as np

# Load Data
df = pd.read_csv('master_dataset.csv')
df.dropna(inplace=True)

# Mapping Label (Sesuai dengan ai_service.py)
intent_map = {"inquiry": 0, "order": 1, "complaint": 2, "other": 3}
sentiment_map = {"positive": 0, "neutral": 1, "negative": 2}

df['intent_label'] = df['intent'].map(intent_map)
df['sentiment_label'] = df['sentiment'].map(sentiment_map)

# Split 80% Train, 20% Val
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

class IndoBERTDataset(Dataset):
    def __init__(self, texts, intents, sentiments):
        self.texts = texts
        self.intents = intents
        self.sentiments = sentiments

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors="pt")
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'intent_labels': torch.tensor(self.intents[idx], dtype=torch.long),
            'sentiment_labels': torch.tensor(self.sentiments[idx], dtype=torch.long)
        }

train_dataset = IndoBERTDataset(train_df['text'].values, train_df['intent_label'].values, train_df['sentiment_label'].values)
val_dataset = IndoBERTDataset(val_df['text'].values, val_df['intent_label'].values, val_df['sentiment_label'].values)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

print(f"Data Training: {len(train_dataset)} | Data Validasi: {len(val_dataset)}")


In [ ]:
# 4. Definisi Arsitektur Multi-Task IndoBERT
import torch.nn as nn

class MultiTaskIndoBERT(nn.Module):
    def __init__(self, model_name="indobenchmark/indobert-base-p1"):
        super(MultiTaskIndoBERT, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.sentiment_classifier = nn.Linear(self.bert.config.hidden_size, 3) # 3 Kelas Sentimen
        self.intent_classifier = nn.Linear(self.bert.config.hidden_size, 4)    # 4 Kelas Intensi

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        
        sentiment_logits = self.sentiment_classifier(pooled_output)
        intent_logits = self.intent_classifier(pooled_output)
        
        return sentiment_logits, intent_logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan perangkat: {device}")
model = MultiTaskIndoBERT().to(device)


In [ ]:
# 5. Training Loop
from torch.optim import AdamW
from tqdm.auto import tqdm

epochs = 5
optimizer = AdamW(model.parameters(), lr=3e-5)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        intent_labels = batch['intent_labels'].to(device)
        sentiment_labels = batch['sentiment_labels'].to(device)
        
        sentiment_logits, intent_logits = model(input_ids, attention_mask)
        
        loss_sentiment = criterion(sentiment_logits, sentiment_labels)
        loss_intent = criterion(intent_logits, intent_labels)
        loss = loss_sentiment + loss_intent
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Loss Training: {total_loss/len(train_loader):.4f}")

    # Validasi Cepat
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            intent_labels = batch['intent_labels'].to(device)
            sentiment_labels = batch['sentiment_labels'].to(device)
            
            s_logits, i_logits = model(input_ids, attention_mask)
            val_loss += criterion(s_logits, sentiment_labels).item() + criterion(i_logits, intent_labels).item()
    print(f"Loss Validasi: {val_loss/len(val_loader):.4f}\n")


In [ ]:
# 6. Export ke ONNX & Simpan Tokenizer
import os
import shutil

OUTPUT_DIR = "emotext_model"
os.makedirs(f"{OUTPUT_DIR}/tokenizer", exist_ok=True)

# Save Tokenizer
tokenizer.save_pretrained(f"{OUTPUT_DIR}/tokenizer")

# Export to ONNX (CPU Mode karena inferensi ONNX di production pakai CPU)
model.to('cpu')
model.eval()

dummy_text = "tes 123"
inputs = tokenizer(dummy_text, padding='max_length', truncation=True, max_length=128, return_tensors="pt")
dummy_input_ids = inputs['input_ids']
dummy_attention_mask = inputs['attention_mask']

onnx_path = f"{OUTPUT_DIR}/multitask_indobert.onnx"
torch.onnx.export(
    model, 
    (dummy_input_ids, dummy_attention_mask), 
    onnx_path,
    input_names=['input_ids', 'attention_mask'],
    output_names=['sentiment_logits', 'intent_logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size'},
        'attention_mask': {0: 'batch_size'},
        'sentiment_logits': {0: 'batch_size'},
        'intent_logits': {0: 'batch_size'}
    },
    opset_version=14
)

print(f"✅ Export ONNX Selesai: {onnx_path}")


In [ ]:
# 7. Zip File & Download
# Akan mendownload file emotext_model.zip yang langsung bisa diekstrak ke folder produk_extension Anda.
!zip -r emotext_model.zip emotext_model
from google.colab import files
files.download('emotext_model.zip')
print("Model sedang di-download! Silakan ekstrak dan ganti file di folder models/indobert_onnx/ Anda.")
